# Stage 3, step 2: token-level embeddings for the BioLangFusion encoders

Stage 1 stored one mean-pooled vector per sequence. The BioLangFusion heads read the whole
token sequence, so this step runs the three frozen encoders over the rows
[step 1](01_blf_datasets.ipynb) selected and keeps every token's final hidden state.

**This notebook is a job, not an analysis.** It loads models and writes tens of gigabytes,
so it is saved without outputs and is meant to be run one benchmark at a time. Nothing here
interprets a result; the first result comes in step 3, when the reproduced heads are scored.

The cache it writes is resumable. Work is done in shards of a few hundred sequences, each
written atomically, and a rerun skips the shards already on disk. If the machine sleeps,
the kernel dies, or you stop it on purpose, run the cell again and it continues.


---

## Setup

### Imports

As in Stage 1, `PYTORCH_ENABLE_MPS_FALLBACK` is set before PyTorch is imported so that any
operation the Apple GPU backend lacks runs on the CPU. The path block finds
`Code/shared_code/` from this notebook's folder.


In [1]:
# Standard library
import os
from pathlib import Path

# Run any operation the Apple GPU backend lacks on the CPU; must precede importing torch
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Data handling and pretrained encoders
import numpy as np
import pandas as pd
import torch

# Shared project code in Code/shared_code; this notebook lives in Code/stage3
import sys
CODE_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "shared_code").is_dir()), Path.cwd() / "Code")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from shared_code import datasets, encoders
from shared_code.fusion import token_embeddings as tokens

### Settings

`BENCHMARK` selects one of the four registered benchmarks; change it and rerun the notebook
for each. The three encoders are BioLangFusion's own, frozen and unchanged: Phase B swaps
come later and write their own caches.

`CHUNK_SIZE` sets how many sequences go in one shard. Smaller shards mean less lost work if
a run is interrupted and a smaller peak in memory; larger shards mean fewer files. A few
hundred suits a laptop.

The cache lives in `Code/stage3/embeddings/<benchmark>/<encoder>/`, which `.gitignore`
already excludes, so nothing here is ever committed.


In [2]:
# Benchmarks to extract, smallest first, so a mistake costs minutes rather than a night.
# The whole list runs unattended; the cache is resumable, so a stopped run continues.
TO_RUN = ["cov_vac_degradation", "fungal_expression", "ecoli_proteins", "mrna_stability"]
BENCHMARK = TO_RUN[0]               # the one the smoke test and timing cells use
ENCODER_KEYS = ["ntv2_100m", "rnafm", "esm2_8m"]   # BioLangFusion's own three, frozen
CHUNK_SIZE = 256        # sequences per shard
DTYPE = np.float16      # halves the cache; frozen features do not need float32
CACHE_ROOT = Path("embeddings")   # Code/stage3/embeddings/, ignored by Git

DEVICE = encoders.select_device()
ROWS = datasets.blf_benchmark_rows(BENCHMARK)
ENCODERS = [encoders.ENCODERS[k] for k in ENCODER_KEYS]
print(f"device {DEVICE}; cache root {CACHE_ROOT.resolve()}")
for key in TO_RUN:
    print(f"  {datasets.DATASETS[key].label}: {len(datasets.blf_benchmark_rows(key))} sequences")

device mps; cache root /Users/julianmckinley/Desktop/Masters/CapstoneCourse/Mina/Code/stage3/embeddings
  CoV-Vac degradation: 2400 sequences
  Fungal expression: 3138 sequences
  E. coli protein expression (low, medium, high): 4450 sequences
  mRNA stability: 23929 sequences


### What this will cost

The token count follows from the sequence lengths and each encoder's tokenization: one token
per nucleotide for RNA-FM, one per codon for ESM-2, one per six nucleotides for Nucleotide
Transformer v2. So the size of the cache is known before any model loads.

Run the cell below to see it for your selection. Across all four benchmarks the total is
about 34 GB in float16, and mRNA stability is 26 GB of that. The other three come to 8 GB
together, which is why they are the ones to do first.


In [3]:
# Token counts, and so cache size, follow from the sequence lengths alone.
def cost_table(keys):
    rows = []
    for key in keys:
        seqs = datasets.blf_benchmark_rows(key)["sequence"].tolist()
        row = {"benchmark": datasets.DATASETS[key].label, "sequences": len(seqs)}
        total = 0
        for encoder in ENCODERS:
            n = sum(tokens.expected_tokens(encoder, s) for s in seqs)
            width = {"ntv2_100m": 512, "rnafm": 640, "esm2_8m": 320}[encoder.key]  # confirmed at load
            size = n * width * np.dtype(DTYPE).itemsize / 1e9
            row[f"{encoder.key} GB"] = round(size, 2)
            total += size
        rows.append({**row, "total GB": round(total, 2)})
    return pd.DataFrame(rows).set_index("benchmark")

COST = cost_table(datasets.BLF_BENCHMARKS)
print(f"selected: {COST.loc[datasets.DATASETS[BENCHMARK].label, 'total GB']} GB"
      f"   all four: {COST['total GB'].sum():.1f} GB")
COST

selected: 0.33 GB   all four: 34.1 GB


,sequences,ntv2_100m GB,rnafm GB,esm2_8m GB,total GB
benchmark,,,,,
Fungal expression,3138,0.35,2.56,0.43,3.33
"E. coli protein expression (low, medium, high)",4450,0.45,3.36,0.56,4.37
mRNA stability,23929,2.71,20.07,3.34,26.12
CoV-Vac degradation,2400,0.04,0.25,0.04,0.33


---

## A smoke test before the long run

Three things can be wrong in a way that only shows up after hours of work: an encoder could
truncate an input, the stored token count could disagree with the sequence, or the protein
stream could be the wrong length for the codon grid. The next cell extracts a handful of
sequences and checks all three, in under a minute.

The last of those checks deserves a note. Stage 1 translated a coding sequence up to its
first stop codon, which is right for a pooled protein vector. It is wrong here: the codon
grid needs exactly one protein token per codon, and that rule gives one fewer for a clean
coding sequence and far fewer for a fragment with an internal stop. On CoV-Vac, whose
81-nucleotide pieces are cut from a construct rather than being coding sequences, it gives a
full reading frame for only 0.7% of rows and nothing at all for 6.2%.

So the protein stream here uses `grid_protein`, which translates every complete codon and
writes stop codons as `X`. That is deviation 4 from step 1, now settled. BioLangFusion does
not say how it fed such fragments to ESM-2, so this choice is ours and is recorded.


In [4]:
# Eight sequences through every encoder, checking what a long run cannot afford to get wrong.
sample = ROWS["sequence"].tolist()[:8]
checks = []
for encoder in ENCODERS:
    texts = tokens.inputs_for_rows(encoder, sample)
    tokenizer, model = encoders.load_encoder(encoder, DEVICE)
    try:
        model.eval()
        states = [tokens.token_states(t, tokenizer, model, encoder.max_len, DTYPE) for t in texts]
    finally:
        del model, tokenizer
    want = [tokens.expected_tokens(encoder, s) for s in sample]
    got = [s.shape[0] for s in states]
    grid = [len(t) == len(s) // 3 for t, s in zip(texts, sample)] if encoder.modality == "protein" else [True] * len(sample)
    checks.append({
        "encoder": encoder.label,
        "width": states[0].shape[1],
        "dtype": str(states[0].dtype),
        "tokens (first 3)": str(got[:3]),
        "token counts match": "ok" if got == want else f"MISMATCH want {want[:3]} got {got[:3]}",
        "protein is one token per codon": "ok" if all(grid) else "MISMATCH",
    })
pd.DataFrame(checks).set_index("encoder")

W1016 10:18:06.391000 48915 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] RnaFmModel LOAD REPORT from: multimolecule/rnafm
Key                                 | Status     | 
------------------------------------+------------+-
ss_head.decoder.weight              | UNEXPECTED | 
lm_head.transform.dense.weight      | UNEXPECTED | 
ss_head.decoder.bias                | UNEXPECTED | 
lm_head.transform.layer_norm.weight | UNEXPECTED | 
lm_head.bias                        | UNEXPECTED | 
lm_head.transform.layer_norm.bias   | UNEXPECTED | 
lm_head.transform.dense.bias        | UNEXPECTED | 
pooler.dense.bias                   | MISSING    | 
pooler.dense.weight                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,width,dtype,tokens (first 3),token counts match,protein is one token per codon
encoder,,,,,
NT v2 100M multi-species,512,float16,"[16, 16, 16]",ok,ok
RNA-FM,640,float16,"[81, 81, 81]",ok,ok
ESM-2 8M,320,float16,"[27, 27, 27]",ok,ok


Every row should report `ok`. A truncation error means the rows being
embedded are not the ones step 1 selected. A token-count mismatch means an encoder tokenizes
differently from the rate `expected_token_count` assumes, which would need checking before
the grid can rely on it.


## How long the full run will take, measured

The honest answer to "how long" is that it depends on your machine, so measure it rather
than guess. The cell below embeds a sample from each encoder, times it, and projects every
benchmark from its token count.

Two things the projection cannot capture. Attention cost grows faster than linearly with
sequence length, so a benchmark of long sequences runs a little slower per token than a
sample of short ones suggests; and the first call to each encoder includes loading weights
from disk, which the timing excludes by running a warm-up pass first.

Extraction runs one sequence at a time. That is simple and exactly right, but it leaves GPU
throughput on the table, since a batch of padded sequences would use the hardware better. If
the projection below comes back longer than you want to sit through, batching is the fix, and
it is worth doing before mRNA stability rather than after.

In [5]:
import time

SAMPLE_N = 24          # sequences to time per encoder
timings = []
for encoder in ENCODERS:
    texts = tokens.inputs_for_rows(encoder, ROWS["sequence"].tolist()[:SAMPLE_N])
    tokenizer, model = encoders.load_encoder(encoder, DEVICE)
    try:
        model.eval()
        tokens.token_states(texts[0], tokenizer, model, encoder.max_len, DTYPE)  # warm up
        start = time.perf_counter()
        n_tokens = sum(tokens.token_states(t, tokenizer, model, encoder.max_len, DTYPE).shape[0]
                       for t in texts)
        elapsed = time.perf_counter() - start
    finally:
        del model, tokenizer
    timings.append({"encoder": encoder.key, "seconds": elapsed, "tokens": n_tokens,
                    "ms per sequence": round(1000 * elapsed / len(texts), 1),
                    "seconds per 1k tokens": round(1000 * elapsed / n_tokens, 3)})
RATE = pd.DataFrame(timings).set_index("encoder")

# Project each benchmark from its token count at the measured rate.
projection = []
for key in datasets.BLF_BENCHMARKS:
    seqs = datasets.blf_benchmark_rows(key)["sequence"].tolist()
    minutes = {}
    for encoder in ENCODERS:
        n = sum(tokens.expected_tokens(encoder, s) for s in seqs)
        minutes[encoder.key] = n / 1000 * RATE.loc[encoder.key, "seconds per 1k tokens"] / 60
    projection.append({"benchmark": datasets.DATASETS[key].label, "sequences": len(seqs),
                       **{f"{k} min": round(v) for k, v in minutes.items()},
                       "total minutes": round(sum(minutes.values())),
                       "total hours": round(sum(minutes.values()) / 60, 1)})
PROJECTION = pd.DataFrame(projection).set_index("benchmark")
print(RATE[["ms per sequence", "seconds per 1k tokens"]].to_string())
print(f"\nprojected total for all four benchmarks: {PROJECTION['total hours'].sum():.1f} hours")
PROJECTION

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] RnaFmModel LOAD REPORT from: multimolecule/rnafm
Key                                 | Status     | 
------------------------------------+------------+-
ss_head.decoder.weight              | UNEXPECTED | 
lm_head.transform.dense.weight      | UNEXPECTED | 
ss_head.decoder.bias                | UNEXPECTED | 
lm_head.transform.layer_norm.weight | UNEXPECTED | 
lm_head.bias                        | UNEXPECTED | 
lm_head.transform.layer_norm.bias   | UNEXPECTED | 
lm_head.transform.dense.bias        | UNEXPECTED | 
pooler.dense.bias                   | MISSING    | 
pooler.dense.weight                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


           ms per sequence  seconds per 1k tokens
encoder                                          
ntv2_100m             12.6                  0.786
rnafm                  9.2                  0.113
esm2_8m                7.5                  0.277

projected total for all four benchmarks: 1.9 hours


,sequences,ntv2_100m min,rnafm min,esm2_8m min,total minutes,total hours
benchmark,,,,,,
Fungal expression,3138,4,4,3,11,0.2
"E. coli protein expression (low, medium, high)",4450,6,5,4,15,0.2
mRNA stability,23929,35,30,24,88,1.5
CoV-Vac degradation,2400,1,0,0,1,0.0


---

## Extract

This is the long cell. It works through `TO_RUN` in order, finishing one benchmark before
starting the next and printing a line per shard, so progress is visible, a complete benchmark
is usable before the rest are done, and an interruption costs at most one shard.

Set it going and leave it. Before you do, two things about the machine: plug it in, and stop
it sleeping. On a Mac, open a terminal and run `caffeinate -i`, leaving that window open until
the run finishes. That prevents idle sleep but not sleep from closing the lid, so leave the lid
open too.


In [6]:
# The long cell. Resumable: rerun after an interruption and it continues where it stopped.
# Benchmarks finish one at a time, so a complete cache is usable before the rest are done.
import datetime

for key in TO_RUN:
    rows = datasets.blf_benchmark_rows(key)
    print(f"=== {datasets.DATASETS[key].label}: {len(rows)} sequences "
          f"({datetime.datetime.now():%H:%M:%S}) ===")
    for encoder in ENCODERS:
        cache_dir = CACHE_ROOT / key / encoder.key
        texts = tokens.inputs_for_rows(encoder, rows["sequence"].tolist())
        cache, computed = tokens.extract(encoder, texts, cache_dir, DEVICE,
                                         chunk_size=CHUNK_SIZE, dtype=DTYPE)
        print(f"  {encoder.label}: {'extracted' if computed else 'already complete'}, "
              f"{len(cache)} sequences, {cache.meta['total_tokens']:,} tokens, "
              f"width {cache.width}, {cache.meta['bytes'] / 1e9:.2f} GB")
    print(f"=== done {datetime.datetime.now():%H:%M:%S} ===\n")

=== CoV-Vac degradation: 2400 sequences (10:18:23) ===
    ntv2_100m: shard 1/10 (1/10 this run)
    ntv2_100m: shard 2/10 (2/10 this run)
    ntv2_100m: shard 3/10 (3/10 this run)
    ntv2_100m: shard 4/10 (4/10 this run)
    ntv2_100m: shard 5/10 (5/10 this run)
    ntv2_100m: shard 6/10 (6/10 this run)
    ntv2_100m: shard 7/10 (7/10 this run)
    ntv2_100m: shard 8/10 (8/10 this run)
    ntv2_100m: shard 9/10 (9/10 this run)
    ntv2_100m: shard 10/10 (10/10 this run)
  NT v2 100M multi-species: extracted, 2400 sequences, 38,400 tokens, width 512, 0.04 GB


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] RnaFmModel LOAD REPORT from: multimolecule/rnafm
Key                                 | Status     | 
------------------------------------+------------+-
ss_head.decoder.weight              | UNEXPECTED | 
lm_head.transform.dense.weight      | UNEXPECTED | 
ss_head.decoder.bias                | UNEXPECTED | 
lm_head.transform.layer_norm.weight | UNEXPECTED | 
lm_head.bias                        | UNEXPECTED | 
lm_head.transform.layer_norm.bias   | UNEXPECTED | 
lm_head.transform.dense.bias        | UNEXPECTED | 
pooler.dense.bias                   | MISSING    | 
pooler.dense.weight                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    rnafm: shard 1/10 (1/10 this run)
    rnafm: shard 2/10 (2/10 this run)
    rnafm: shard 3/10 (3/10 this run)
    rnafm: shard 4/10 (4/10 this run)
    rnafm: shard 5/10 (5/10 this run)
    rnafm: shard 6/10 (6/10 this run)
    rnafm: shard 7/10 (7/10 this run)
    rnafm: shard 8/10 (8/10 this run)
    rnafm: shard 9/10 (9/10 this run)
    rnafm: shard 10/10 (10/10 this run)
  RNA-FM: extracted, 2400 sequences, 194,400 tokens, width 640, 0.25 GB


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

...

[transformers] RnaFmModel LOAD REPORT from: multimolecule/rnafm
Key                                 | Status     | 
------------------------------------+------------+-
ss_head.decoder.weight              | UNEXPECTED | 
lm_head.transform.dense.weight      | UNEXPECTED | 
ss_head.decoder.bias                | UNEXPECTED | 
lm_head.transform.layer_norm.weight | UNEXPECTED | 
lm_head.bias                        | UNEXPECTED | 
lm_head.transform.layer_norm.bias   | UNEXPECTED | 
lm_head.transform.dense.bias        | UNEXPECTED | 
pooler.dense.bias                   | MISSING    | 
pooler.dense.weight                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    rnafm: shard 1/94 (1/94 this run)
    rnafm: shard 2/94 (2/94 this run)
    rnafm: shard 3/94 (3/94 this run)
    rnafm: shard 4/94 (4/94 this run)
    rnafm: shard 5/94 (5/94 this run)
    rnafm: shard 6/94 (6/94 this run)
    rnafm: shard 7/94 (7/94 this run)
    rnafm: shard 8/94 (8/94 this run)
    rnafm: shard 9/94 (9/94 this run)
    rnafm: shard 10/94 (10/94 this run)
    rnafm: shard 11/94 (11/94 this run)
    rnafm: shard 12/94 (12/94 this run)
    rnafm: shard 13/94 (13/94 this run)
    rnafm: shard 14/94 (14/94 this run)
    rnafm: shard 15/94 (15/94 this run)
    rnafm: shard 16/94 (16/94 this run)
    rnafm: shard 17/94 (17/94 this run)
    rnafm: shard 18/94 (18/94 this run)
    rnafm: shard 19/94 (19/94 this run)
    rnafm: shard 20/94 (20/94 this run)
    rnafm: shard 21/94 (21/94 this run)
    rnafm: shard 22/94 (22/94 this run)
    rnafm: shard 23/94 (23/94 this run)
    rnafm: shard 24/94 (24/94 this run)
    rnafm: shard 25/94 (25/94 this run)
    rnafm: shard 2

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    esm2_8m: shard 1/94 (1/94 this run)
    esm2_8m: shard 2/94 (2/94 this run)
    esm2_8m: shard 3/94 (3/94 this run)
    esm2_8m: shard 4/94 (4/94 this run)
    esm2_8m: shard 5/94 (5/94 this run)
    esm2_8m: shard 6/94 (6/94 this run)
    esm2_8m: shard 7/94 (7/94 this run)
    esm2_8m: shard 8/94 (8/94 this run)
    esm2_8m: shard 9/94 (9/94 this run)
    esm2_8m: shard 10/94 (10/94 this run)
    esm2_8m: shard 11/94 (11/94 this run)
    esm2_8m: shard 12/94 (12/94 this run)
    esm2_8m: shard 13/94 (13/94 this run)
    esm2_8m: shard 14/94 (14/94 this run)
    esm2_8m: shard 15/94 (15/94 this run)
    esm2_8m: shard 16/94 (16/94 this run)
    esm2_8m: shard 17/94 (17/94 this run)
    esm2_8m: shard 18/94 (18/94 this run)
    esm2_8m: shard 19/94 (19/94 this run)
    esm2_8m: shard 20/94 (20/94 this run)
    esm2_8m: shard 21/94 (21/94 this run)
    esm2_8m: shard 22/94 (22/94 this run)
    esm2_8m: shard 23/94 (23/94 this run)
    esm2_8m: shard 24/94 (24/94 this run)
    esm2_8

## Verify the finished cache

The check that matters is per row, not in aggregate: every sequence must have exactly the
tokens its length implies for that encoder. An aggregate byte count would hide a single
sequence that was silently truncated or dropped.


In [7]:
# Per-row verification: every sequence must have exactly the tokens its length implies.
report = []
for key in TO_RUN:
    rows = datasets.blf_benchmark_rows(key)
    for encoder in ENCODERS:
        cache_dir = CACHE_ROOT / key / encoder.key
        if not (cache_dir / "meta.json").exists():
            report.append({"benchmark": datasets.DATASETS[key].label, "encoder": encoder.label,
                           "all rows match": "not extracted yet"})
            continue
        cache = tokens.summarize(cache_dir)
        want = np.array([tokens.expected_tokens(encoder, s) for s in rows["sequence"]])
        matches = bool(len(cache) == len(rows) and np.array_equal(np.asarray(cache.lengths), want))
        report.append({
            "benchmark": datasets.DATASETS[key].label,
            "encoder": encoder.label,
            "rows": f"{len(cache)} / {len(rows)}",
            "all rows match": matches,
            "tokens": f"{cache.meta['total_tokens']:,}",
            "width": cache.width,
            "GB on disk": round(sum(p.stat().st_size for p in cache_dir.glob("shard_*.npz")) / 1e9, 2),
        })
VERIFY = pd.DataFrame(report).set_index(["benchmark", "encoder"])
print("every row matches:", all(r.get("all rows match") is True for r in report))
VERIFY

every row matches: True


rows  \
benchmark                                      encoder                                   
CoV-Vac degradation                            NT v2 100M multi-species    2400 / 2400   
                                               RNA-FM                      2400 / 2400   
                                               ESM-2 8M                    2400 / 2400   
Fungal expression                              NT v2 100M multi-species    3138 / 3138   
                                               RNA-FM                      3138 / 3138   
                                               ESM-2 8M                    3138 / 3138   
E. coli protein expression (low, medium, high) NT v2 100M multi-species    4450 / 4450   
                                               RNA-FM                      4450 / 4450   
                                               ESM-2 8M                    4450 / 4450   
mRNA stability                                 NT v2 100M multi-species  23929 / 23929   
                                               RNA-FM                    23929 / 23929   
                                               ESM-2 8M                  23929 / 23929   

                                                                         all rows match  \
benchmark                                      encoder                                    
CoV-Vac degradation                            NT v2 100M multi-species            True   
                                               RNA-FM                              True   
                                               ESM-2 8M                            True   
Fungal expression                              NT v2 100M multi-species            True   
                                               RNA-FM                              True   
                                               ESM-2 8M                            True   
E. coli protein expression (low, medium, high) NT v2 100M multi-species            True   
                                               RNA-FM                              True   
                                               ESM-2 8M                            True   
mRNA stability                                 NT v2 100M multi-species            True   
                                               RNA-FM                              True   
                                               ESM-2 8M                            True   

                                                                             tokens  \
benchmark                                      encoder                                
CoV-Vac degradation                            NT v2 100M multi-species      38,400   
                                               RNA-FM                       194,400   
                                               ESM-2 8M                      64,800   
Fungal expression                              NT v2 100M multi-species     337,376   
                                               RNA-FM                     2,000,436   
                                               ESM-2 8M                     666,812   
E. coli protein expression (low, medium, high) NT v2 100M multi-species     442,550   
                                               RNA-FM                     2,622,015   
                                               ESM-2 8M                     874,005   
mRNA stability                                 NT v2 100M multi-species   2,642,703   
                                               RNA-FM                    15,678,663   
                                               ESM-2 8M                   5,226,221   

                                                                         width  \
benchmark                                      encoder                           
CoV-Vac degradation                            NT v2 100M multi-species    512   
                                               RNA-FM                      640   

With `all rows match` true for all three encoders, this benchmark is
ready for step 3. Record the run in [docs/run-records.md](../../docs/run-records.md) with the
date, device, and the printed sizes, as Stage 1 runs were recorded.


---

## Order of work, and what step 3 needs

Do the benchmarks in this order, smallest first, so a mistake costs minutes rather than a
night:

1. `cov_vac_degradation`, about 0.3 GB. Proves the pipeline end to end.
2. `fungal_expression`, about 3.4 GB. This is the benchmark step 3 checks the build against,
   because its published numbers are clean: 0.804 for the best single encoder, 0.824 fused,
   and no sequence appears in more than one split.
3. `ecoli_proteins`, about 4.3 GB. The one classification benchmark.
4. `mrna_stability`, about 26 GB. Leave it running overnight.

Step 3 then reads these caches and builds the BioLangFusion pipeline on top: the codon-grid
resampling, the three fusion heads, and the fixed TextCNN head. It needs from here, per
benchmark and per encoder, a token cache whose rows line up by position with
`blf_benchmark_rows(benchmark)`, which is what the verification above establishes.

One thing step 3 must decide and this step deliberately does not: how a stream whose tokens
do not divide evenly into the grid is resampled. Nucleotide Transformer v2 gives one token
per six nucleotides, which is two codons, so a sequence whose codon count is odd leaves a
half token at the end. BioLangFusion's transposed convolution assumes an exact factor of two.
That is a modelling choice, so it belongs with the model.
